In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle

class TextoClassificadorDNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(TextoClassificadorDNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes) 
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# Carregar o conversor de palavras para números (o teu TF-IDF)
with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

df_subm = pd.read_csv('subm1.csv', sep=';')
textos_para_prever = df_subm['Text'] 

X_novos_numeros = vectorizer.transform(textos_para_prever).toarray()
X_novos_tensor = torch.tensor(X_novos_numeros, dtype=torch.float32)

print(f"Dados carregados! Temos {X_novos_tensor.shape[0]} frases para prever.")

Dados carregados! Temos 150 frases para prever.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_size = X_novos_tensor.shape[1]
modelo_torch = TextoClassificadorDNN(input_size=input_size, num_classes=5).to(device)

modelo_torch.load_state_dict(torch.load('melhor_modelo_pytorch.pth', map_location=device))

# 3. Trancar o modelo no modo de Avaliação (desliga o Dropout para ele focar a 100%)
modelo_torch.eval()

# Fazer as previsões reais
with torch.no_grad():
    X_novos_tensor = X_novos_tensor.to(device)
    outputs = modelo_torch(X_novos_tensor)
    _, previsoes = torch.max(outputs.data, 1)

# 5. Adicionar a coluna das previsões ao ficheiro do professor
df_subm['label'] = previsoes.cpu().numpy()

nome_ficheiro_entrega = 'subm1-g6-MIA-B.csv'
df_subm.to_csv(nome_ficheiro_entrega, index=False, sep=';')

print(f"Sucesso absoluto! O ficheiro {nome_ficheiro_entrega} foi gerado e está pronto a entregar.")

Sucesso absoluto! O ficheiro subm1-g6-MIA-B.csv foi gerado e está pronto a entregar.
